In [6]:
import pandas as pd

df_housing = pd.read_csv("/content/california_housing.csv")
display(df_housing.head())

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,median_house_value
0,-122.23,37.88,41,880,129.0,322,126,8.3252,NEAR BAY,452600
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,NEAR BAY,358500
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,NEAR BAY,352100
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,NEAR BAY,341300
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,NEAR BAY,342200


In [7]:
# Make a copy to avoid modifying the original DataFrame
df = df_housing.copy()

# Create new features
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population'] / df['households']

# Display the new features and basic info
display(df[['rooms_per_household', 'bedrooms_per_room', 'population_per_household']].head())
print(df.info())

,rooms_per_household,bedrooms_per_room,population_per_household
0,6.984127,0.146591,2.555556
1,6.238137,0.155797,2.109842
2,8.288136,0.129516,2.802260
3,5.817352,0.184458,2.547945
4,6.281853,0.172096,2.181467


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   longitude                 20640 non-null  float64
 1   latitude                  20640 non-null  float64
 2   housing_median_age        20640 non-null  int64  
 3   total_rooms               20640 non-null  int64  
 4   total_bedrooms            20433 non-null  float64
 5   population                20640 non-null  int64  
 6   households                20640 non-null  int64  
 7   median_income             20640 non-null  float64
 8   ocean_proximity           20640 non-null  object 
 9   median_house_value        20640 non-null  int64  
 10  rooms_per_household       20640 non-null  float64
 11  bedrooms_per_room         20433 non-null  float64
 12  population_per_household  20640 non-null  float64
dtypes: float64(7), int64(5), object(1)
memory usage: 2.0+ MB
None

In [8]:
# Check for missing values
print("Missing values before handling:\n", df.isnull().sum())

# Fill missing 'total_bedrooms' with the median
df['total_bedrooms'].fillna(df['total_bedrooms'].median(), inplace=True)

# Recalculate 'bedrooms_per_room' after filling missing 'total_bedrooms'
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']

print("\nMissing values after handling:\n", df.isnull().sum())

Missing values before handling:
 longitude                     0
latitude                      0
housing_median_age            0
total_rooms                   0
total_bedrooms              207
population                    0
households                    0
median_income                 0
ocean_proximity               0
median_house_value            0
rooms_per_household           0
bedrooms_per_room           207
population_per_household      0
dtype: int64

Missing values after handling:
 longitude                   0
latitude                    0
housing_median_age          0
total_rooms                 0
total_bedrooms              0
population                  0
households                  0
median_income               0
ocean_proximity             0
median_house_value          0
rooms_per_household         0
bedrooms_per_room           0
population_per_household    0
dtype: int64


/tmp/ipykernel_1914/3283004166.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['total_bedrooms'].fillna(df['total_bedrooms'].median(), inplace=True)


In [9]:
import numpy as np

# Simple outlier removal based on domain knowledge or statistical methods
# For example, removing houses with unusually high bedrooms per room or population per household
# Using IQR for `bedrooms_per_room` as an example

Q1 = df['bedrooms_per_room'].quantile(0.25)
Q3 = df['bedrooms_per_room'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out outliers
df_filtered = df[(df['bedrooms_per_room'] >= lower_bound) & (df['bedrooms_per_room'] <= upper_bound)]

print(f"Original dataset size: {len(df)} rows")
print(f"Dataset size after removing outliers: {len(df_filtered)} rows")

df = df_filtered.copy()

Original dataset size: 20640 rows
Dataset size after removing outliers: 20005 rows


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Separate features (X) and target (y)
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

# Identify numerical and categorical features
numerical_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

# Create a preprocessing pipeline for numerical features (StandardScaler)
numerical_transformer = StandardScaler()

# Create a preprocessing pipeline for categorical features (OneHotEncoder)
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# First, split into training (70%) and temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

# Then, split temp (30%) into validation (20% of total) and test (10% of total)
# 20% of total means (2/3) of temp; 10% of total means (1/3) of temp
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=1/3, random_state=42)

print(f"Training set size: {len(X_train)} samples")
print(f"Validation set size: {len(X_val)} samples")
print(f"Test set size: {len(X_test)} samples")

# Apply preprocessing separately to each set to avoid data leakage
# Note: fit_transform only on training data, transform on validation and test data
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

Training set size: 14003 samples
Validation set size: 4001 samples
Test set size: 2001 samples


In [11]:
from sklearn.tree import DecisionTreeRegressor

# Initialize and train the Decision Tree Regressor
# We use a Regressor because 'median_house_value' is continuous, not categorical.
# If classification was strictly required, 'median_house_value' would need to be binned into categories.
dtree_regressor = DecisionTreeRegressor(random_state=42)
dtree_regressor.fit(X_train_processed, y_train)

print("Decision Tree Regressor trained successfully.")

Decision Tree Regressor trained successfully.


In [12]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score

# Predict on the validation set
y_pred_val = dtree_regressor.predict(X_val_processed)

# Evaluate the model using regression metrics
mse = mean_squared_error(y_val, y_pred_val)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_val, y_pred_val)
r2 = r2_score(y_val, y_pred_val)

print("### Validation Set Metrics ###")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R-squared (R2): {r2:.2f}")

print("\n--- Note on Classification Metrics ---")
print("Accuracy, F1-score, and Recall are metrics for classification problems.")
print("Since 'median_house_value' is a continuous target, we use regression metrics (MSE, RMSE, MAE, R2).")
print("If you intended to categorize house prices for classification, the target variable would need to be binned.")

# Perform Cross-Validation
# We will use the full training data (X_train_processed, y_train) for cross-validation
# The validation set was used for hyperparameter tuning decisions (not done yet, but typically).

print("\n### Cross-Validation ###")
cv_scores = cross_val_score(dtree_regressor, X_train_processed, y_train, cv=5, scoring='neg_mean_squared_error')
rmse_cv_scores = np.sqrt(-cv_scores)

print(f"Cross-validation RMSE scores: {rmse_cv_scores}")
print(f"Mean CV RMSE: {rmse_cv_scores.mean():.2f}")
print(f"Standard deviation of CV RMSE: {rmse_cv_scores.std():.2f}")

### Validation Set Metrics ###
Mean Squared Error (MSE): 4387112206.54
Root Mean Squared Error (RMSE): 66235.28
Mean Absolute Error (MAE): 43001.74
R-squared (R2): 0.67

--- Note on Classification Metrics ---
Accuracy, F1-score, and Recall are metrics for classification problems.
Since 'median_house_value' is a continuous target, we use regression metrics (MSE, RMSE, MAE, R2).
If you intended to categorize house prices for classification, the target variable would need to be binned.

### Cross-Validation ###
Cross-validation RMSE scores: [69842.43444579 67500.30157179 69110.3872575  69748.6153808
 69750.42212496]
Mean CV RMSE: 69190.43
Standard deviation of CV RMSE: 884.67


In [13]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid to search
# 'max_depth' controls overfitting (deeper trees can overfit)
# 'min_samples_leaf' controls overfitting (smaller values can overfit)
param_grid = {
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 4, 8, 16]
}

# Initialize the Decision Tree Regressor
dtree_base = DecisionTreeRegressor(random_state=42)

# Initialize GridSearchCV
# We'll use the training data for cross-validation within GridSearchCV
# scoring='neg_mean_squared_error' is used because GridSearchCV maximizes scores,
# and MSE is a loss function (lower is better), so we negate it.
grid_search = GridSearchCV(
    dtree_base,
    param_grid,
    cv=5, # 5-fold cross-validation
    scoring='neg_mean_squared_error',
    n_jobs=-1, # Use all available CPU cores
    verbose=1
)

# Fit GridSearchCV on the preprocessed training data
grid_search.fit(X_train_processed, y_train)

# Get the best parameters and best score
best_params = grid_search.best_params_
best_rmse_cv = np.sqrt(-grid_search.best_score_)

print(f"Best Hyperparameters: {best_params}")
print(f"Best Cross-validated RMSE: {best_rmse_cv:.2f}")

# Get the best model
best_dtree_model = grid_search.best_estimator_

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best Hyperparameters: {'max_depth': 15, 'min_samples_leaf': 16}
Best Cross-validated RMSE: 58539.46


In [14]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Predict on the unseen test set using the best model
y_pred_test = best_dtree_model.predict(X_test_processed)

# Evaluate the tuned model's performance on the test set
mse_test = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
r2_test = r2_score(y_test, y_pred_test)

print("### Tuned Model Test Set Metrics ###")
print(f"Mean Squared Error (MSE): {mse_test:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_test:.2f}")
print(f"Mean Absolute Error (MAE): {mae_test:.2f}")
print(f"R-squared (R2): {r2_test:.2f}")

print("\nCompare these test set metrics to the validation metrics from the initial model and the cross-validated RMSE.")
print("Improvements in these metrics, especially a lower RMSE and higher R2 on the test set,")
print("indicate that hyperparameter tuning has likely helped in reducing overfitting and improving generalization.")

### Tuned Model Test Set Metrics ###
Mean Squared Error (MSE): 3193314629.27
Root Mean Squared Error (RMSE): 56509.42
Mean Absolute Error (MAE): 37508.00
R-squared (R2): 0.76

Compare these test set metrics to the validation metrics from the initial model and the cross-validated RMSE.
Improvements in these metrics, especially a lower RMSE and higher R2 on the test set,
indicate that hyperparameter tuning has likely helped in reducing overfitting and improving generalization.


In [15]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

print("### Demonstrating Underfitting (Very Simple Decision Tree) ###")

# Train a Decision Tree with intentionally low max_depth to cause underfitting
dtree_underfit = DecisionTreeRegressor(max_depth=3, random_state=42)
dtree_underfit.fit(X_train_processed, y_train)

# Predict on training and test sets
y_train_pred_underfit = dtree_underfit.predict(X_train_processed)
y_test_pred_underfit = dtree_underfit.predict(X_test_processed)

# Evaluate training set performance
mse_train_underfit = mean_squared_error(y_train, y_train_pred_underfit)
rmse_train_underfit = np.sqrt(mse_train_underfit)
r2_train_underfit = r2_score(y_train, y_train_pred_underfit)

# Evaluate test set performance
mse_test_underfit = mean_squared_error(y_test, y_test_pred_underfit)
rmse_test_underfit = np.sqrt(mse_test_underfit)
r2_test_underfit = r2_score(y_test, y_test_pred_underfit)

print(f"\nTraining Set RMSE (Underfit): {rmse_train_underfit:.2f}")
print(f"Training Set R2 (Underfit): {r2_train_underfit:.2f}")
print(f"Test Set RMSE (Underfit): {rmse_test_underfit:.2f}")
print(f"Test Set R2 (Underfit): {r2_test_underfit:.2f}")

print("\nAs you can see, both training and test set R2 scores are significantly lower than our tuned model (R2 ~0.76).")
print("This indicates the model is too simple and is underfitting the data.")

### Demonstrating Underfitting (Very Simple Decision Tree) ###

Training Set RMSE (Underfit): 74109.23
Training Set R2 (Underfit): 0.59
Test Set RMSE (Underfit): 74163.89
Test Set R2 (Underfit): 0.59

As you can see, both training and test set R2 scores are significantly lower than our tuned model (R2 ~0.76).
This indicates the model is too simple and is underfitting the data.
